(chap_m2s5)=
# Explainability

**Content creators:** [Renee](https://github.com/reneeandreea)

:::{admonition} Chapter Overview
:class: tip

* Some models are **interpretable** by construction
* Interpretability degrades fast with complexity: a shallow decision tree is readable, a forest of hundreds of trees is not
* **Explainability** methods such as SHAP approximate *why* an opaque model made a given prediction, after the fact
* SHAP explains the **model** but it does not explain the **disease**--a confidently explained bad model is still a bad model
* Explanations are a bias-detection tool, not only a trust-building one
:::

:::{admonition} Important: this dataset is synthetic
:class: danger

This chapter reuses the **synthetic cardiovascular cohort** built in [Fairness and Bias in Model Evaluation](../M2S4/M2S4nb.ipynb). No real patients are involved, and the dataset contains a bias we deliberately engineered for teaching purposes.

None of the risk relationships shown here are epidemiological findings. Do not quote the feature importances, SHAP values, or subgroup differences on this page as facts about cardiovascular disease. See the M2S4 chapter for a full description of how the data was generated and what was built into it.
:::

## Getting Started

In the previous chapter we found that our cardiovascular risk model performed badly for one subgroup of patients. We diagnosed *that* it failed, and we understood *why* only because we had written the data-generating code ourselves.

In a real model, you will have to analyze predictions. This chapter asks the follow-up question:

**How can we determine how a model makes its predictions?**

:::{admonition} Clinical Question
:class: info
* **What are we trying to understand?** Which features drive a model's risk predictions? We are interested in considering this question for both the whole cohort and for one individual patient.
* **Why does it matter clinically?** A clinician being asked to act on a risk score needs to know what produced it. "The model said so" is not a basis for a treatment decision, and it is not something you can discuss with a patient.
* **Why does it matter for fairness?** As we will see at the end of this chapter, looking inside a model can tell you *why* it fails a subgroup. Performance metrics alone cannot.
:::

## Setting Up

The cell below loads the libraries for this page. Alongside the usual tools it imports `shap`, the library implementing the explanation method we use later, and scikit-learn's tree models. Expand the cell for more details.

In [ ]:
"""
These are imports that help the rest of the code on this page run.
"""

from matplotlib import pyplot as plt
from myst_nb import glue
import numpy as np
import pandas as pd
import shap
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree

The next cell rebuilds the exact synthetic cohort from the fairness chapter, so the two chapters are directly comparable. It is the same generating code, reproduced here so this page runs on its own. Expand it to see how the data are made, or read the [fairness chapter](../M2S4/M2S4nb.ipynb) for the full explanation of the bias we engineered.

In [ ]:
"""
Rebuilds the synthetic cohort from the M2S4 fairness chapter.
This is invented data with a deliberately engineered bias.
This code section was written with the support of generative AI.
"""

rng = np.random.default_rng(21)
n = 8000

race = rng.choice(["White", "Black", "South Asian", "East Asian"],
                  size=n, p=[0.70, 0.10, 0.11, 0.09])
sex = np.where(rng.random(n) < 0.38, "Female", "Male")

age = rng.normal(58, 12, n).clip(30, 90)
smoker = rng.binomial(1, 0.22, n)
systolic_bp = rng.normal(128, 16, n).clip(90, 200)
cholesterol = rng.normal(5.2, 1.0, n).clip(2.5, 9.0)
bmi = rng.normal(27.5, 4.5, n).clip(16, 50)

# The engineered difference: Black women have lower cholesterol, higher BP,
# and their risk is driven by blood pressure rather than cholesterol.
is_bw = (race == "Black") & (sex == "Female")
cholesterol = np.where(is_bw, (cholesterol - 1.10).clip(2.5, 9.0), cholesterol)
systolic_bp = np.where(is_bw, (systolic_bp + 12).clip(90, 200), systolic_bp)

logit = np.where(
    is_bw,
    -29.6 + 0.14*age + 0.13*systolic_bp + 0.15*cholesterol + 0.088*bmi + 1.40*smoker,
    -26.6 + 0.14*age + 0.055*systolic_bp + 1.55*cholesterol + 0.088*bmi + 1.40*smoker,
)

patients = pd.DataFrame({
    "age": age.round(0).astype(int), "sex": sex, "race": race,
    "systolic_bp": systolic_bp.round(0), "cholesterol": cholesterol.round(1),
    "bmi": bmi.round(1), "smoker": smoker,
    "cvd_event": rng.binomial(1, 1 / (1 + np.exp(-logit))),
})

FEATURES = ["age", "systolic_bp", "cholesterol", "bmi", "smoker"]
train_set, test_set = train_test_split(
    patients, test_size=0.3, random_state=0, stratify=patients["cvd_event"]
)
test_set = test_set.copy()
print(f"Training patients: {len(train_set)}   Test patients: {len(test_set)}")

## Interpretability vs. Explainability

These two terms are often used interchangeably, but in fact, they describe different properties.

**Interpretability** is a property of the **model itself**. An interpretable model is transparent by construction--you can inspect it and read out the rule it applies. The logistic regression from the fundamentals chapter is interpretable: its coefficients *are* the model. A shallow decision tree is interpretable: the tree *is* the rule.

**Explainability** is a property of an **explanation you attach afterwards**. When a model is too complex to read directly, you can build a second, simpler model (or other tool) that approximates its behaviour, and inspect this instead. SHAP works this way.

A rough test: if someone asks *"why did the model say that?"* and you answer by pointing at the model, it is interpretable. If you answer by running another tool first, you are practicing explainability.

:::{admonition} Why do we care about this difference?
:class: important
An interpretable model tells you what it does. An explanation tells you what some other tool *estimates* the model does.

Post-hoc explanations are approximations, and approximations can be wrong or incomplete. The ethical modelling literature suggests that for high-stakes decisions--including clinical decisions--the right move is to use an inherently interpretable model rather than an opaque model with an explanation bolted onto it {cite:p}`rudin_2019`.

This chapter shows you both, and then gives you a concrete reason to aim for interpretability.
:::

## An Interpretable Model: The Decision Tree

A decision tree splits patients by asking a sequence of yes/no questions. Each patient follows one path from the top of the tree to a leaf, and the leaf gives the prediction.

We deliberately limit ours to a depth of three, so the whole model fits on a page.

In [ ]:
tree = DecisionTreeClassifier(max_depth=3, random_state=0)
tree.fit(train_set[FEATURES], train_set["cvd_event"])

test_set["pred_tree"] = tree.predict(test_set[FEATURES])
tree_accuracy = accuracy_score(test_set["cvd_event"], test_set["pred_tree"])
print(f"Decision tree accuracy: {tree_accuracy*100:.1f}%")

In [ ]:
fig, ax = plt.subplots(figsize=(16, 8))
plot_tree(tree, feature_names=FEATURES, class_names=["No event", "Event"],
          filled=True, rounded=True, fontsize=9, ax=ax)
ax.set_title("A depth-3 decision tree: the model is the diagram")
plt.tight_layout()
plt.show()

glue("fig_tree", fig, display=False)

```{glue:figure} fig_tree
:align: center
:name: fig-tree

A depth-3 decision tree. Every prediction the model makes can be traced by following a single path from top to bottom.
```

You can read this. Start at the top, answer each question about your patient, and arrive at a prediction. Nothing is hidden, and no separate tool is needed.

The same model prints as text, which is often easier to follow:

In [ ]:
print(export_text(tree, feature_names=FEATURES))

**This is an interpretable model.** The model *is* a human-readable rule. If a clinician asks why a patient was flagged, you trace the path: *this patient is over 59, their cholesterol is above 5.05, so the model predicts an event.*

Notice something else: the tree tells you which features it uses at all. Only three of our five features appear anywhere. This is an easily observable finding.

## Where We Lose the Plot (pun intended)

Shallow trees are readable but blunt. The standard fix is a **random forest**: train hundreds of trees, each on a different random slice of the data and a random subset of features, then allow for consensus.

In [ ]:
forest = RandomForestClassifier(n_estimators=300, random_state=0, n_jobs=-1)
forest.fit(train_set[FEATURES], train_set["cvd_event"])

test_set["pred_forest"] = forest.predict(test_set[FEATURES])
forest_accuracy = accuracy_score(test_set["cvd_event"], test_set["pred_forest"])

print(f"Decision tree accuracy:  {tree_accuracy*100:.1f}%")
print(f"Random forest accuracy:  {forest_accuracy*100:.1f}%")

depths = [t.get_depth() for t in forest.estimators_]
leaves = [t.get_n_leaves() for t in forest.estimators_]
print(f"\nTrees in the forest: {len(forest.estimators_)}")
print(f"Average tree depth:  {np.mean(depths):.1f}")
print(f"Total leaves across the whole forest: {sum(leaves):,}")

The forest is more accurate, but it is also no longer something a person can read.

To trace one prediction by hand you would follow a path through **300 separate trees**, each around 22 levels deep, and tally the votes--close to 300,000 leaves in total. The forest still applies a perfectly definite rule, we just cannot expect a single clinician to read, interpret, and explain this rule.

:::{admonition} The trade-off
:class: note
Accuracy went up. Interpretability is almost zero.

This is the tension we are interested in resolving in this chapter, and it is the reason post-hoc explainability tools exist at all. If every good model were readable, we would not need SHAP.
:::

Tree-based models do offer one built-in summary--**feature importance**, which measures roughly how much each feature reduced prediction error across the forest:

In [ ]:
importances = pd.DataFrame({
    "feature": FEATURES,
    "importance": forest.feature_importances_.round(3),
}).sort_values("importance", ascending=False)

print(importances.to_string(index=False))

This is a start, but it is underdeveloped. It tells us which features mattered *on average, across all patients*. It does not tell us:

* **Which direction** a feature pushed the prediction. Does high cholesterol raise or lower predicted risk?
* **How much** it pushed, in units we can interpret.
* **Why this particular patient** was flagged in the first place.

For that we need something better.

## SHAP Values

SHAP stands for **SHapley Additive exPlanations** {cite:p}`lundberg_2017`. The idea it borrows from is older than machine learning, coming from cooperative game theory.

### The idea behind Shapley values

Imagine a team that works together to produce some outcome. Each member contributed, but they worked jointly, so you cannot simply identify who did what. How do you divide the credit fairly?

Lloyd Shapley's (1951) answer follows: 

*For any one member, consider **every possible order** in which the team could have assembled. Each iteration, measure how much the outcome improved at the moment that member joined. Average this improvement across all possible orderings. This average is their fair share of the credit.*

The reason to average over orderings is that contributions are not independent. A member who is valuable to an empty team may add little to a team that already covers their skills. Averaging over every arrangement stops the answer from depending on some arbitrary order.

SHAP maps this directly onto a prediction:

* The **team** is the set of features
* The **outcome** is the model's prediction for one patient
* The **baseline** is what the model predicts knowing nothing (i.e., roughly the average prediction across the data)
* Each feature's **SHAP value** is its fair share of the distance between that baseline and this patient's actual prediction

This produces the property that makes SHAP useful:

> **baseline prediction + sum of all SHAP values = this patient's prediction**

The contributions add up exactly. Nothing is left unexplained, and nothing is double-counted.

:::{admonition} Why we need a special algorithm
:class: note
Averaging over every possible ordering is resource-expensive. With 5 features there are only 32 subsets of features to consider, but the work grows as $2^{k}$ for $k$ features. A 20-feature model means over a million subsets **per patient**, and a real clinical model may have hundreds of features!

Computing SHAP values exactly by brute force is therefore not practical. `TreeExplainer` exploits the structure of tree-based models to compute them **exactly** in time that grows polynomially rather than exponentially, making SHAP usable on a forest. For models that are not tree-based, `shap` falls back on sampling-based approximations.
:::

In [ ]:
# TreeExplainer computes exact SHAP values for tree-based models
explainer = shap.TreeExplainer(forest)

# Take a sample of the test set
sample = test_set.sample(800, random_state=0)
shap_values = explainer.shap_values(sample[FEATURES])

# For a binary classifier, keep the contributions toward the positive class
if isinstance(shap_values, list):
    shap_values = shap_values[1]
elif shap_values.ndim == 3:
    shap_values = shap_values[:, :, 1]

print(f"SHAP values computed for {shap_values.shape[0]} patients "
      f"across {shap_values.shape[1]} features")

### Checking that the contributions really do add up

Before trusting any of this, it is worth verifying the additivity property on a single patient rather than taking it on faith.

In [ ]:
baseline = explainer.expected_value
if isinstance(baseline, (list, np.ndarray)) and np.size(baseline) > 1:
    baseline = np.asarray(baseline).ravel()[1]
baseline = float(baseline)

one_patient = 0
reconstructed = baseline + shap_values[one_patient].sum()
actual = forest.predict_proba(sample[FEATURES])[one_patient, 1]

print(f"Baseline (average prediction):     {baseline:.4f}")
print(f"Sum of this patient's SHAP values: {shap_values[one_patient].sum():+.4f}")
print(f"Baseline + SHAP values =           {reconstructed:.4f}")
print(f"Model's actual prediction =        {actual:.4f}")

They match. Every unit of predicted risk has been attributed to a specific feature!

### The summary plot

Applying this across many patients at once gives us a picture of the model's overall behaviour that is far richer than the feature-importance table.

In [ ]:
fig = plt.figure()
shap.summary_plot(shap_values, sample[FEATURES], show=False)
plt.title("SHAP summary: how each feature moves predicted risk")
plt.tight_layout()
plt.show()

glue("fig_shap_summary", fig, display=False)

```{glue:figure} fig_shap_summary
:align: center
:name: fig-shap-summary

Each dot is one patient. Horizontal position shows how much that feature moved their predicted risk; colour shows whether the feature value was high or low.
```

:::{admonition} How to read this plot
:class: info

* **Each dot is one patient**, and there is one row per feature.
* **Horizontal position is the SHAP value.** Dots to the right of centre are patients whose predicted risk was pushed *up* by that feature; dots to the left were pushed *down*. Distance from centre is how strongly.
* **Colour is the feature's value** for that patient. Red for high, blue for low.
* **Rows are ordered by overall influence**, most important at the top.

Now read the pattern. For `cholesterol`, the red dots sit on the right and the blue dots on the left: high cholesterol pushes predicted risk up, low cholesterol pushes it down, consistently. This clean colour split indicated a simple increasing relationship.

**Vertical spread carries information too.** Where dots of the same colour fan out horizontally, the same feature value affected different patients by different amounts, meaning the model has learned an *interaction*, where a feature's effect depends on the other features. A plain logistic regression cannot represent this kind of interaction.
:::

### Explaining a single patient

The payoff is per-patient. We can decompose one prediction into its composite parts.

In [ ]:
# Pick the patient the model considers highest risk
probabilities = forest.predict_proba(sample[FEATURES])[:, 1]
idx = int(np.argmax(probabilities))
patient = sample.iloc[idx]

contributions = pd.DataFrame({
    "feature": FEATURES,
    "patient_value": [patient[f] for f in FEATURES],
    "shap_contribution": shap_values[idx].round(4),
}).sort_values("shap_contribution", key=abs, ascending=False)

print("Patient being explained:")
print(f"  age={patient['age']}, systolic_bp={patient['systolic_bp']}, "
      f"cholesterol={patient['cholesterol']}, bmi={patient['bmi']}, "
      f"smoker={patient['smoker']}")
print(f"  predicted risk: {probabilities[idx]:.3f}")
print(f"  actual outcome: {patient['cvd_event']}\n")
print(contributions.to_string(index=False))

In [ ]:
# The same explanation as a chart
fig, ax = plt.subplots(figsize=(8, 4))
ordered = contributions.iloc[::-1]
bar_colors = ["#c0392b" if v > 0 else "#2c7fb8" for v in ordered["shap_contribution"]]
ax.barh(ordered["feature"], ordered["shap_contribution"], color=bar_colors)
ax.axvline(0, color="#333333", linewidth=1)
ax.set_xlabel("SHAP contribution to predicted risk")
ax.set_title(f"Why the model flagged this patient (predicted risk {probabilities[idx]:.2f})")
plt.tight_layout()
plt.show()

glue("fig_one_patient", fig, display=False)

```{glue:figure} fig_one_patient
:align: center
:name: fig-one-patient

A single patient's prediction broken into feature contributions. Red bars pushed predicted risk up; blue bars pushed it down.
```

This is the kind of output that a clinician can interpret. It shows a clinician that *this patient is high risk*, but more importantly, it shows them that *this patient is high risk mainly because of their age, with blood pressure and cholesterol adding to the effect, and not being a smoker diminishing the effect.*

This is an actionable statement: a clinician can agree with it, challenge it, or act on it otherwise.

## Connecting Back to Fairness

So far we have used explanations to build confidence in the model. Now we can use them for something more interesting.

In the [fairness chapter](../M2S4/M2S4nb.ipynb) we found that our logistic regression missed nearly 60% of cardiovascular events among Black women. We knew why only because we had written the data-generating code. Let us see whether SHAP could have told us instead.

Recall the mechanism we engineered: for most of the cohort risk is driven by **cholesterol**, but for Black women it is driven by **blood pressure**. If the forest has partly learned this, SHAP should show blood pressure doing the heavy lifting for that particular subgroup.

In [ ]:
is_black_woman = (sample["race"] == "Black") & (sample["sex"] == "Female")

comparison = pd.DataFrame({
    "feature": FEATURES,
    "mean_abs_shap_black_women": np.abs(shap_values[is_black_woman.values]).mean(axis=0).round(4),
    "mean_abs_shap_everyone_else": np.abs(shap_values[~is_black_woman.values]).mean(axis=0).round(4),
})
comparison["ratio"] = (comparison["mean_abs_shap_black_women"]
                       / comparison["mean_abs_shap_everyone_else"]).round(2)

print(f"Black women in sample: {is_black_woman.sum()}   "
      f"Everyone else: {(~is_black_woman).sum()}\n")
print(comparison.to_string(index=False))

Blood pressure carries noticeably more weight for Black women than for everyone else, while the other features are much closer. The forest has partially discovered the pattern we built in, and SHAP surfaced it **without anyone needing to know the ground truth in advance**.

This is a compelling finding. Subgroup performance metrics tell you *that* a model fails a group. Explanations can begin to tell you *what the model is doing differently* for that group, which is the first step toward knowing how to fix it!

We can look at this finding more directly by plotting how the model responds to blood pressure for each group:

In [ ]:
bp_index = FEATURES.index("systolic_bp")
fig, ax = plt.subplots(figsize=(8, 5))

ax.scatter(sample.loc[~is_black_woman, "systolic_bp"],
           shap_values[~is_black_woman.values, bp_index],
           s=12, alpha=0.35, color="#8fa8bf", label="Everyone else")
ax.scatter(sample.loc[is_black_woman, "systolic_bp"],
           shap_values[is_black_woman.values, bp_index],
           s=42, alpha=0.9, color="#c0392b", label="Black women")

ax.axhline(0, color="#333333", linewidth=1)
ax.set_xlabel("Systolic blood pressure")
ax.set_ylabel("SHAP contribution of blood pressure")
ax.set_title("How much blood pressure moves the model's prediction")
ax.legend()
plt.tight_layout()
plt.show()

glue("fig_bp_dependence", fig, display=False)

```{glue:figure} fig_bp_dependence
:align: center
:name: fig-bp-dependence

Each dot is one patient. Black women sit further to the right (i.e., for higher blood pressure) and blood pressure contributes more to their predictions.
```

### The uncomfortable result

Having built all this explanatory machinery around the forest, it is worth asking whether **the forest is actually the model we should be using instead.**

Let us compare all three models on the fairness metric from the previous chapter.

In [ ]:
# Refit the logistic regression from the fairness chapter for comparison
logreg = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
logreg.fit(train_set[FEATURES], train_set["cvd_event"])
test_set["pred_logreg"] = logreg.predict(test_set[FEATURES])

model_columns = {
    "logistic regression": "pred_logreg",
    "decision tree (depth 3)": "pred_tree",
    "random forest (300)": "pred_forest",
}

rows = []
for (race_value, sex_value), subset in test_set.groupby(["race", "sex"]):
    row = {"group": f"{race_value} / {sex_value}"}
    for label, column in model_columns.items():
        row[label] = round(recall_score(subset["cvd_event"], subset[column],
                                        zero_division=0), 3)
    rows.append(row)

recall_table = pd.DataFrame(rows).sort_values("logistic regression")
print("RECALL BY SUBGROUP\n")
print(recall_table.to_string(index=False))

print("\n\nOVERALL ACCURACY vs EQUAL OPPORTUNITY GAP\n")
for label, column in model_columns.items():
    accuracy = accuracy_score(test_set["cvd_event"], test_set[column])
    gap = recall_table[label].max() - recall_table[label].min()
    print(f"  {label:<26} accuracy={accuracy:.3f}   equal-opportunity gap={gap:.3f}")

Let's carefully consider what happened.

The **decision tree is the least accurate model**, showing about 75% accuracy, which puts it five points behind the logistic regression. In a standard model-selection process it would be discarded immediately.

It is also, by a wide margin, **the fairest**. Its recall gap between the best- and worst-served subgroups is roughly half that of the other two models, and it detects substantially more of the cardiovascular events among Black women than either the logistic regression or the forest.

:::{admonition} Why did the simplest model do best for the worst-served group?
:class: dropdown

Because a tree can split.

A logistic regression fits **one coefficient per feature for everybody**. It must find a single compromise between a cholesterol-mediated rule and a blood-pressure-mediated one, and since the large majority of patients are cholesterol-mediated, the compromise lands close to them.

A decision tree can branch: it can apply one rule to patients above a blood pressure threshold and a different rule below it. Our depth-3 tree splits on `systolic_bp` in several places, which happens to catch part of the group whose risk is blood-pressure-driven.

Two important caveats:

* **This is not a general law.** Decision trees are not inherently fair, and simple models are not inherently fair. What happened here is that this particular model's structure happened to align with this particular data's structure. On different data the ranking could reverse entirely!
* **The forest can split too**, and its SHAP values show that it did, if only partly. But a forest is fitted to minimize *overall* error, and a group making up 3.5% of the data contributes very little to that objective. Thus, the capacity to represent the subgroup exists without much pressure to use it.

The transferable lesson is not "use trees." It is that **the model that wins on aggregate accuracy is not automatically the model that serves everyone best**, and you cannot know which is which unless you measure both.
:::

### What explanations cannot do

One final word of caution, and it is the **most important point in this chapter.**

**SHAP explains the model. It does not explain the disease.**

Every value in this section describes what our forest learned from our synthetic data. If a model has learned a relationship that is wrong (e.g., because the training data was unrepresentative, or because a feature is a proxy for something we did not intend) SHAP will faithfully and confidently report that wrong relationship. It has no access to biology, and no way to tell a real mechanism from an artefact of how the data was collected.

This is exactly what happened in the Obermeyer study from the previous chapter {cite:p}`obermeyer_2019`. An explanation of that algorithm would have correctly reported that prior healthcare cost drove its predictions. That explanation would have been perfectly accurate about the model, and would still have missed the point entirely, because the problem was that cost was the *wrong* feature to predict in the first place.

:::{admonition} A confident explanation of a bad model is still a bad model
:class: warning
Explanations make models feel trustworthy. This feeling alone is *not* good evidence.

An explanation tells you what the model is doing. Whether what it is doing is *right* is a clinical question, and answering it requires domain knowledge, subgroup evaluation, and prospective testing. SHAP is a merely a lens, not a verdict.
:::

## Summary

We took the synthetic cardiovascular cohort from the fairness chapter and deconstructed three models.

A **depth-3 decision tree** was fully interpretable: the model was a diagram you could read and trace by hand. A **300-tree random forest** was more accurate and completely unreadable, with close to 300,000 leaves. To say anything at all about why it predicted what it did, we needed **SHAP**, which distributes credit for each prediction across the features using an idea from cooperative game theory, in a way that confidently adds up.

SHAP then allowed us to reveal *why* the model treated one subgroup differently. It showed us that blood pressure carried more weight in its predictions for Black women. Subgroup metrics told us the model failed them. The explanation began to tell us what it was doing instead.

And the comparison at the end gave us a valuable result: the least accurate model was the fairest; the most accurate model was not. **Accuracy and fairness are different questions, and a model-selection process that only asks the first will overlook the second.**

:::{admonition} A practical checklist
:class: tip
When thinking about explainability for a clinical model:

1. **Try an interpretable model first.** If a shallow tree or a regression performs acceptably, you may not need post-hoc explanation at all (and you get transparency for free!).
2. **Be explicit about which one you are doing.** "This model is interpretable" and "we generated an explanation for this model" are different claims with different guarantees.
3. **Report global and per-patient explanations.** Feature importance describes the model; clinicians need to know about the patient in front of them.
4. **Use explanations to interrogate subgroup failures**, not only to build confidence. They are a debugging tool.
5. **Remember that SHAP explains the model, not the biology.** Validate the relationship; do not just admire the plot!
6. **Do not let an explanation substitute for evaluation.** A well-explained model that has never been tested on a subgroup or validated prospectively is still an untested model.
:::

Together with the [fairness chapter](../M2S4/M2S4nb.ipynb), this gives you two complementary tools: subgroup evaluation tells you **whether** a model is failing someone, and explainability begins to tell you **why**.